In [ ]:
%reset -f

##### Hardware initialization

In [ ]:
# MCL Z positioner

from hardware.mclController import MCLStage, loadMCLDLL, ProductInformation
import time

print("Initializing Stage")
try: # if stage was already initialized, shut it down first
    stage.shutDown()
    time.sleep(0.5)
except NameError:
    pass

# DLL file location
mcl_lib = "c:/Program Files/Mad City Labs/NanoDrive/Madlib"  

stage = MCLStage(mcl_lib)
if not stage.getStatus():
    exit()

print("Stage Properties:")
stageProperties = stage.getProperties()
for key in sorted(stageProperties):
        print(key, '\t', stageProperties[key])
print("")

print("Z axis range:", "%.2f um" % stage.getAxisRange(3))
print("")

zPosition = 50 # initial z position in microns
stage.zMoveTo(zPosition)
time.sleep(0.05) # let it move, minimum wait for 0.05s

stage.getPosition(3)
print("Current Z Position:", "%.4f um" %stage.getPosition(3))
print("")

In [ ]:
# SPAD512 camera
# open the software first

from hardware.SPAD512S import SPAD512S
import pySPADutils as util

port = 9990 # Check the command Server in the setting tab of the software and change it if necessary
SPAD1 = SPAD512S(port)

info = SPAD1.get_info()
print("\nGeneral informations of the camera :")
print(info)
temp = SPAD1.get_temps() # Current temperatures of FPGAs, PCB and Chip
print("\nCurrent temperatures of FPGAs, PCB and Chip :")
print(temp)
freq = SPAD1.get_freq() # Operating frequencies (Laser and frame)
print("\nOperating frequencies (Laser and frame) :")
print(freq)
SPAD1.enable_cooling(1) # Enable the cooling system of the camera

##### Acquization parameter setup

In [ ]:
# Camera settings
overlap = 1
timeout = 0
pileup = 1

im_width = 512
rowCol = 512 #Number of rows/columns
bitdepth = 1

# Each layer
BP = 5000
intTime = 20 # in us if 1-bit

# Z scan settings, 
# mode 0: single layer
# mode 1: Z stack, +/- Z_range around Z_middle, with step size of Z_stepSize
# mode 2: Z stack, from Z_middle to Z_middle + Z_range, with step size of Z_stepSize
# mode 3: Z stack, 1:3 depth around Z_middle, with step size of Z_stepSize
# mode 4: Z stack, 1:7 depth around Z_middle, with step size of Z_stepSize
Z_mode = 0 
Z_stepSize = 0.1    # in microns
Z_range    = 2      # in microns

Z_middle = 50 
match Z_mode:
    case 0:
        Z_positions = [Z_middle]
    case 1:
        n_steps = int(Z_range/Z_stepSize) + 1
        Z_positions = [Z_middle + (i-n_steps//2)*Z_stepSize for i in range(n_steps)]
    case 2:
        n_steps = int(Z_range/Z_stepSize) + 1
        Z_positions = [Z_middle + i*Z_stepSize for i in range(n_steps)]
    case 3:
        n_steps = int(Z_range/Z_stepSize) + 1
        Z_positions = [Z_middle + (i-n_steps//3)*Z_stepSize for i in range(n_steps)]
    case 4:
        n_steps = int(Z_range/Z_stepSize) + 1
        Z_positions = [Z_middle + (i-n_steps//7)*Z_stepSize for i in range(n_steps)]

print("Z positions to scan:", Z_positions, "microns")

##### Exposure

In [ ]:
from pytictoc import TicToc
from tqdm import tqdm

t = TicToc()
data = []
zbar = tqdm(Z_positions)
t.tic()
for z in zbar:
    zbar.set_description(f"Z @ {z:.2f} um")

    # Move stage to the z position
    stage.zMoveTo(z)
    time.sleep(0.05)
    
    # Acquire image at the z position
    data = SPAD1.get_intensity_1bit_packed(BP, intTime, bitdepth, overlap, timeout, pileup, im_width)
    time.sleep(0.05)


t.toc("Acquisition time took")
zbar.close()

In [ ]:
filePath = "test.bin"
util.writeBinBig(filePath, data)

##### Close stage handle

In [ ]:
stage.shutDown()